# Variants TMT Dataset — Checkpoint 4: Building the Classifier

Goal: train a first model to predict **Severe-COVID-19** vs **Non-severe-COVID-19** using the top 200 statistically ranked peptides saved at the end of Checkpoint 3.

This checkpoint intentionally uses `checkpoint4_modeling_input_top200.tsv` rather than the FDR-significant peptide table, because the FDR-filtered table was empty.

## Step 1 — Load the Checkpoint 4 modeling input

The input table should contain one row per patient, one `Condition` column, and 200 peptide intensity columns selected in Checkpoint 3.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

INPUT_FILE = Path("checkpoint4_modeling_input_top200.tsv")

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        "checkpoint4_modeling_input_top200.tsv was not found. "
        "Run the final Checkpoint 3 save cell first, or copy that TSV into this folder."
    )

df = pd.read_csv(INPUT_FILE, sep="\t", index_col=0)

condition = df["Condition"]
X = df.drop(columns=["Condition"])

print(f"Input file: {INPUT_FILE}")
print(f"Patients: {X.shape[0]}")
print(f"Peptides/features: {X.shape[1]}")
print("\nCondition counts:")
print(condition.value_counts())

df.head()

Input file: checkpoint4_modeling_input_top200.tsv
Patients: 43
Peptides/features: 200

Condition counts:
Condition
Non-severe-COVID-19    25
Severe-COVID-19        18
Name: count, dtype: int64


,Condition,"R.[304.207]LTG(R,-19.035)GAEDSLADQAAN(K,304.207).W","R.{290.173}[304.207]EGT(C,57.021)PEAPTDE(C,57.021)(K,304.207)PV(K,304.207).W","R.[304.207]SG(K,304.207)DPNHFRPAGLPE(K,304.207).Y","R.[304.207]EANYIG(S,304.212)D(K,304.207)YFHAR.G","R.{290.175}[304.207]NE(C,57.021)FLQH(K,304.207)DDNPNLPR.L","R.[304.207](F,-13.032)(K,304.207)DLGEENF(K,304.207).A","R.[304.207](L,285.173)TGRGAEDSLADQAAN(K,304.207).W","K.[304.207](C,348.19)(C,57.021)AAADPHE(C,57.021)YA(K,304.207).V","K.[304.207](V,290.171)DNALQSGNSQESVTEQDS(K,304.207).D",...,"K.[304.207]RPSGVS(N,-16.989)R.F","R.[201.872]AF(K,304.207)AWAVARLSQRFP(K,304.207)AEFAEVS(K,304.207).L","R.{17.033}[304.207]LYGSEAFATDFQDSAAA(K,304.207).K","S.[304.207](Q,-2.127)EEE(K,304.207)TEALTSA(K,304.207).R","K.[304.207]VQFELHYQEV(K,247.146).W","R.[304.207]EANYI(G,303.228)SD(K,304.207)YFHAR.G","R.[304.207]HYEG(S,-18.005)TVPE(K,304.207)(K,304.207).T","K.[304.207](K,595.381)VPQVSTPTLVEVSR.N","K.[304.207](A,291.17)LPAPIE(K,304.207).T","C.[304.207]TAFHDNEET(F,-65.968)L(K,304.207).K"
Non-severe-COVID-19.Patient-group-PT,Non-severe-COVID-19,0.048036,0.628984,0.272482,0.292385,2.322728,1.211855,0.014281,0.393012,1.196619,...,0.288652,1.838134,1.222122,1.508213,1.172871,0.144222,0.420095,0.211338,NaN,1.989289
Non-severe-COVID-19.XG1,Non-severe-COVID-19,0.025385,2.084133,0.052785,0.074656,2.921881,1.182953,0.010519,1.779172,1.797625,...,0.155543,0.552098,0.479762,0.837873,0.955676,0.118804,0.324748,0.902967,NaN,NaN
Non-severe-COVID-19.XG10,Non-severe-COVID-19,0.201227,1.745563,NaN,0.281456,0.792378,0.753542,0.046497,1.527595,0.804133,...,0.109613,0.251221,0.388955,NaN,NaN,NaN,0.605142,0.335603,0.329013,0.915820
Non-severe-COVID-19.XG11,Non-severe-COVID-19,0.243141,2.767648,NaN,0.033040,0.987688,1.278250,0.004799,1.907997,1.299629,...,0.162244,0.426492,0.233199,NaN,NaN,NaN,0.260984,0.443953,0.411119,0.599079
Non-severe-COVID-19.XG13,Non-severe-COVID-19,0.108329,1.967572,NaN,0.057281,1.031713,0.967744,0.008867,1.863626,0.941431,...,0.602384,0.415062,0.318465,NaN,NaN,NaN,0.520331,0.414138,0.505308,1.022561


## Step 2 — Encode the labels

The model needs numeric labels. Here, Severe is encoded as `1` and Non-severe is encoded as `0`.

In [2]:
LABEL_MAP = {
    "Non-severe-COVID-19": 0,
    "Severe-COVID-19": 1,
}

y = condition.map(LABEL_MAP)

if y.isna().any():
    bad_labels = condition[y.isna()].unique().tolist()
    raise ValueError(f"Unexpected condition labels found: {bad_labels}")

print("Label encoding:")
print("0 = Non-severe-COVID-19")
print("1 = Severe-COVID-19")
print("\nEncoded label counts:")
print(y.value_counts().sort_index())

Label encoding:
0 = Non-severe-COVID-19
1 = Severe-COVID-19

Encoded label counts:
Condition
0    25
1    18
Name: count, dtype: int64


## Step 3 — Split into train, validation, and test sets

We keep a held-out test set for Checkpoint 5. In this checkpoint, only the training and validation sets are used for model selection.

In [3]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# First split off 20% as a final test set.
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

# Split the remaining 80% into training and validation.
# 0.25 of 80% = 20% of the full dataset.
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.25,
    stratify=y_trainval,
    random_state=RANDOM_STATE,
)

print(f"Train shape:      {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Test shape:       {X_test.shape}")
print("\nTrain labels:")
print(y_train.value_counts().sort_index())
print("\nValidation labels:")
print(y_val.value_counts().sort_index())
print("\nTest labels:")
print(y_test.value_counts().sort_index())

Train shape:      (25, 200)
Validation shape: (9, 200)
Test shape:       (9, 200)

Train labels:
Condition
0    15
1    10
Name: count, dtype: int64

Validation labels:
Condition
0    5
1    4
Name: count, dtype: int64

Test labels:
Condition
0    5
1    4
Name: count, dtype: int64


## Step 4 — Build preprocessing + model pipelines

Missing peptide intensities are imputed using the training-set median only. Logistic regression also standardizes features because it is sensitive to feature scale.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "logistic_regression_l2": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l2",
            C=1.0,
            class_weight="balanced",
            max_iter=5000,
            random_state=RANDOM_STATE,
        )),
    ]),
    "logistic_regression_l1": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l1",
            C=0.2,
            solver="liblinear",
            class_weight="balanced",
            max_iter=5000,
            random_state=RANDOM_STATE,
        )),
    ]),
    "random_forest": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=500,
            max_depth=3,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),
}

print("Models ready:")
for name in models:
    print("-", name)

Models ready:
- logistic_regression_l2
- logistic_regression_l1
- random_forest


## Step 5 — Train models and compare validation performance

Because the dataset is small, validation scores can move a lot depending on the split. Treat this as a first baseline, not a final performance estimate.

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

comparison_rows = []
fitted_models = {}
validation_predictions = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    fitted_models[name] = pipeline

    y_pred = pipeline.predict(X_val)

    # Some models provide probabilities; use them for ROC-AUC when available.
    if hasattr(pipeline, "predict_proba"):
        y_score = pipeline.predict_proba(X_val)[:, 1]
        roc_auc = roc_auc_score(y_val, y_score)
    else:
        y_score = np.full(len(y_val), np.nan)
        roc_auc = np.nan

    validation_predictions[name] = pd.DataFrame({
        "patient": X_val.index,
        "true_label": y_val.values,
        "predicted_label": y_pred,
        "severe_probability": y_score,
    })

    comparison_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_val, y_pred),
        "precision_severe": precision_score(y_val, y_pred, zero_division=0),
        "recall_severe": recall_score(y_val, y_pred, zero_division=0),
        "f1_severe": f1_score(y_val, y_pred, zero_division=0),
        "roc_auc": roc_auc,
    })

model_comparison = pd.DataFrame(comparison_rows).sort_values(
    ["f1_severe", "roc_auc", "accuracy"],
    ascending=False,
).reset_index(drop=True)

model_comparison

,model,accuracy,precision_severe,recall_severe,f1_severe,roc_auc
0,logistic_regression_l2,1.000000,1.000000,1.0,1.0,1.00
1,random_forest,1.000000,1.000000,1.0,1.0,1.00
2,logistic_regression_l1,0.777778,0.666667,1.0,0.8,0.95


## Step 6 — Select the best baseline model

The selected model is the one with the best validation F1 score for the Severe class. This prioritizes catching Severe patients while still accounting for false positives.

In [6]:
best_model_name = model_comparison.loc[0, "model"]
best_model = fitted_models[best_model_name]

print(f"Best validation model: {best_model_name}")
print("\nValidation metrics:")
display(model_comparison.loc[model_comparison["model"] == best_model_name])

best_validation_predictions = validation_predictions[best_model_name]
best_validation_predictions

Best validation model: logistic_regression_l2

Validation metrics:


,model,accuracy,precision_severe,recall_severe,f1_severe,roc_auc
0,logistic_regression_l2,1.0,1.0,1.0,1.0,1.0


,patient,true_label,predicted_label,severe_probability
0,Severe-COVID-19.XG44,1,1,0.890919
1,Severe-COVID-19.XG34,1,1,0.998726
2,Non-severe-COVID-19.XG3,0,0,0.295500
3,Severe-COVID-19.XG42,1,1,0.992455
4,Non-severe-COVID-19.XG21,0,0,0.107111
5,Non-severe-COVID-19.XG1,0,0,0.018908
6,Non-severe-COVID-19.XG5,0,0,0.016613
7,Non-severe-COVID-19.XG9,0,0,0.026191
8,Severe-COVID-19.XG33,1,1,0.999982


## Step 7 — Save Checkpoint 4 outputs

The test set is saved but not evaluated here. Use it in Checkpoint 5 for final evaluation.

In [9]:
import joblib

OUTPUT_DIR = Path("./models")

# Save split datasets with Condition labels restored for readability.
train_df = pd.concat([y_train.map({0: "Non-severe-COVID-19", 1: "Severe-COVID-19"}).rename("Condition"), X_train], axis=1)
val_df = pd.concat([y_val.map({0: "Non-severe-COVID-19", 1: "Severe-COVID-19"}).rename("Condition"), X_val], axis=1)
test_df = pd.concat([y_test.map({0: "Non-severe-COVID-19", 1: "Severe-COVID-19"}).rename("Condition"), X_test], axis=1)

train_df.to_csv(OUTPUT_DIR / "checkpoint4_train_top200.tsv", sep="\t")
val_df.to_csv(OUTPUT_DIR / "checkpoint4_validation_top200.tsv", sep="\t")
test_df.to_csv(OUTPUT_DIR / "checkpoint4_test_top200.tsv", sep="\t")

model_comparison.to_csv(OUTPUT_DIR / "checkpoint4_model_comparison.tsv", sep="\t", index=False)
best_validation_predictions.to_csv(OUTPUT_DIR / "checkpoint4_validation_predictions.tsv", sep="\t", index=False)
joblib.dump(best_model, OUTPUT_DIR / "checkpoint4_best_model.joblib")

# Save a simple text file recording the selected model name.
(OUTPUT_DIR / "checkpoint4_best_model_name.txt").write_text(best_model_name + "\n")

print("Checkpoint 4 files saved:")
print("- checkpoint4_train_top200.tsv")
print("- checkpoint4_validation_top200.tsv")
print("- checkpoint4_test_top200.tsv")
print("- checkpoint4_model_comparison.tsv")
print("- checkpoint4_validation_predictions.tsv")
print("- checkpoint4_best_model.joblib")
print("- checkpoint4_best_model_name.txt")

Checkpoint 4 files saved:
- checkpoint4_train_top200.tsv
- checkpoint4_validation_top200.tsv
- checkpoint4_test_top200.tsv
- checkpoint4_model_comparison.tsv
- checkpoint4_validation_predictions.tsv
- checkpoint4_best_model.joblib
- checkpoint4_best_model_name.txt


## Summary — Building the Classifier. Refer to `checkpoint4.md`

Checkpoint 4 loads the top 200 peptide modeling input from Checkpoint 3, splits patients into train/validation/test sets, imputes missing values inside each model pipeline, trains three baseline classifiers, chooses the best validation model, and saves the model plus split files for Checkpoint 5.